<a href="https://colab.research.google.com/github/pdkoumudi2510/AI-ML-PROJECTS/blob/main/AI%20Due%20Diligence%20copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
path = kagglehub.dataset_download("pranjalverma08/sec-edgar-annual-financial-filings-2021")

100%|██████████| 62.8M/62.8M [00:00<00:00, 114MB/s]

Extracting files...


In [14]:
# 1. Install zstd first so Ollama can extract its files
!apt-get install zstd -y

# 2. Force install all dependencies cleanly
!pip install streamlit langchain langchain-core langchain-community langchain-chroma langchain-huggingface sentence-transformers python-dotenv kagglehub -q

# 3. Install localtunnel to expose the Streamlit UI safely
!npm install -g localtunnel -q

# 4. Install the Ollama background engine
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [15]:
import kagglehub

# Download the SEC filings dataset
path = kagglehub.dataset_download("pranjalverma08/sec-edgar-annual-financial-filings-2021")
print("Dataset successfully downloaded to cache path:", path)

Using Colab cache for faster access to the 'sec-edgar-annual-financial-filings-2021' dataset.
Dataset successfully downloaded to cache path: /kaggle/input/sec-edgar-annual-financial-filings-2021


In [16]:
import os
import shutil
import subprocess
import time

# 1. Start the Ollama background server
print("Starting Ollama server daemon...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# 2. Pull the local model
print("Downloading local phi3 model... (this may take a minute)")
subprocess.run(["ollama", "pull", "phi3:mini"])
print("🟢 Ollama engine ready with local model phi3!")

# 3. Create active workspace directory
os.makedirs("documents", exist_ok=True)

# 4. Walk through all subdirectories recursively to extract hidden documents
copied_count = 0
for root, dirs, files in os.walk(path):
    for file_name in files:
        if copied_count < 3:
            # Skip hidden metadata files or system items
            if file_name.startswith('.') or file_name.endswith('~'):
                continue

            full_file_path = os.path.join(root, file_name)
            target_path = os.path.join("documents", file_name)

            shutil.copy(full_file_path, target_path)
            print(f"Copied: {file_name} -> documents/")
            copied_count += 1

print(f"\n🟢 Complete! 'documents/' folder now contains {copied_count} real files.")

Starting Ollama server daemon...
🟢 Ollama engine ready with local model phi3!
Copied: 1378590_10K_2021_0001437749-21-028984.json -> documents/
Copied: 1408278_10K_2020_0001408278-21-000050.json -> documents/
Copied: 1066194_10K_2021_0001558370-21-012388.json -> documents/

🟢 Complete! 'documents/' folder now contains 3 real files.


In [17]:
%%writefile app.py
import os
import streamlit as st
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

st.set_page_config(page_title="SEC Edgar AI Due Diligence Copilot", page_icon="📈", layout="wide")
st.title("📈 SEC Edgar AI Due Diligence Copilot (100% Local RAG)")
st.caption("Perform automated due diligence on real SEC filings using open-source, on-device AI.")

DOCS_DIR = "documents"
available_files = os.listdir(DOCS_DIR) if os.path.exists(DOCS_DIR) else []

with st.sidebar:
    st.header("Control Panel")
    st.success("🟢 Connected to local Phi-3 & HuggingFace Models")
    st.markdown("---")
    st.markdown(f"### Indexed SEC Filings ({len(available_files)}):")
    for f in available_files[:5]:
        st.markdown(f"- `{f}`")

# Format documents helper
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

@st.cache_resource(show_spinner="Running local vectorizer and chunking SEC filings...")
def initialize_local_copilot():
    if not os.path.exists(DOCS_DIR) or not os.listdir(DOCS_DIR):
        st.error("No documents found in the 'documents/' folder!")
        return None, None

    all_documents = []
    for file_name in os.listdir(DOCS_DIR):
        file_path = os.path.join(DOCS_DIR, file_name)
        if os.path.isfile(file_path):
            try:
                loader = TextLoader(file_path, encoding='utf-8')
                all_documents.extend(loader.load())
            except:
                try:
                    loader = TextLoader(file_path, encoding='latin-1')
                    all_documents.extend(loader.load())
                except:
                    pass

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    chunks = text_splitter.split_documents(all_documents)

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = Chroma.from_documents(chunks, embeddings)
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    llm = Ollama(model="phi3:mini", temperature=0)

    system_prompt = (
        "<|system|>\n"
        "You are an expert financial analyst examining official SEC Edgar financial filings.\n"
        "Analyze the provided document context thoroughly to answer the user's inquiry. "
        "Keep your output data-driven and extract precise financial numbers exactly as written.\n"
        "Context:\n{context}<|end|>\n"
        "<|user|>\n{input}<|end|>\n<|assistant|>"
    )

    prompt = ChatPromptTemplate.from_template(system_prompt)

    # Modern LCEL Chain Build
    lcel_chain = (
        {"context": retriever | format_docs, "input": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return lcel_chain, retriever

if available_files:
    copilot_chain, retriever_engine = initialize_local_copilot()
else:
    st.warning("Please populate the 'documents/' folder before continuing.")
    copilot_chain, retriever_engine = None, None

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []
if "preset_query" not in st.session_state:
    st.session_state.preset_query = None

st.subheader("⚡ Executive Quick-Actions")
col1, col2, col3 = st.columns(3)

with col1:
    if st.button("🚨 Identify Risk Disclosures"):
        st.session_state.preset_query = "What operational risks, bottlenecks, or uncertainties does the company mention in these filings?"
with col2:
    if st.button("📈 Summarize Revenue & Performance"):
        st.session_state.preset_query = "Summarize the primary financial performance highlights, gross revenues, and income results mentioned."
with col3:
    if st.button("🔍 Audit Balance Sheet Elements"):
        st.session_state.preset_query = "What details are listed regarding cash balances, investments, long-term liabilities, or debt allocations?"

st.markdown("---")

for message in st.session_state.chat_history:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

def run_rag_query(query_text):
    if not copilot_chain:
        return
    with st.chat_message("user"):
        st.markdown(query_text)
    st.session_state.chat_history.append({"role": "user", "content": query_text})

    with st.chat_message("assistant"):
        with st.spinner("Analyzing SEC filings via local weights..."):
            try:
                # Invoke our modernized chain
                answer = copilot_chain.invoke(query_text)

                # Retrieve matching metadata sources safely for citations
                source_docs = retriever_engine.get_relevant_documents(query_text)
                sources = set([os.path.basename(doc.metadata['source']) for doc in source_docs])

                source_footer = f"\n\n**Verified Citations:** SEC Edgar Source File ({', '.join(sources)})"
                full_output = answer + source_footer
                st.markdown(full_output)
                st.session_state.chat_history.append({"role": "assistant", "content": full_output})
            except Exception as e:
                st.error(f"Execution Error: {str(e)}")

if st.session_state.preset_query:
    query_to_run = st.session_state.preset_query
    st.session_state.preset_query = None
    run_rag_query(query_to_run)

if user_input := st.chat_input("Ask a custom question about these SEC filings..."):
    run_rag_query(user_input)

Overwriting app.py


In [ ]:
# 1. Print Tunnel IP Password
print("Your Local Tunnel Password IP:")
!wget -qO- ipv4.icanhazip.com

# 2. Launch Streamlit and pass it directly to localtunnel
!streamlit run app.py & npx localtunnel --port 8501

Your Local Tunnel Password IP:
34.106.206.140
⠙⠹⠸⠼⠴⠦

⠧⠇⠏⠋⠙⠹your url is: https://short-flowers-peel.loca.lt
2026-07-09 20:39:11.853 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.106.206.140:8501

/content/app.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
modules.json: 100% 349/349 [00:00<00:00, 1.35MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 455kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 19.8MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 233kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.62MB/s]
model.safetensors: 100% 90.9M/90.9M [00:01<00:00, 6